# 03 - Modeling

This notebook trains a Logistic Regression baseline, tunes XGBoost with Optuna, evaluates ROC-AUC and F1, plots confusion matrix and ROC curve, and creates SHAP summary and waterfall explanations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import shap
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from evaluate import evaluate_classifier, save_shap_summary, save_shap_waterfall, transformed_features
from preprocess import build_model_pipeline, clean_telco_data, create_dataset_split, load_telco_data, save_pipeline
from train import build_logistic_regression, build_xgboost_classifier, tune_xgboost

DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_modeling"

In [ ]:
raw_df = load_telco_data(DATA_PATH)
clean_df = clean_telco_data(raw_df)
split = create_dataset_split(clean_df, test_size=0.2, random_state=42)
split.X_train.head()

## Logistic Regression Baseline

In [ ]:
baseline_pipeline = build_model_pipeline(
    build_logistic_regression(random_state=42),
    split.X_train,
    random_state=42,
    use_smote=True,
)
baseline_pipeline.fit(split.X_train, split.y_train)
baseline_metrics = evaluate_classifier(baseline_pipeline, split.X_test, split.y_test)
baseline_metrics

## XGBoost with Optuna

In [ ]:
best_params = tune_xgboost(split.X_train, split.y_train, n_trials=25, random_state=42)
best_params

In [ ]:
xgb_pipeline = build_model_pipeline(
    build_xgboost_classifier(params=best_params, random_state=42),
    split.X_train,
    random_state=42,
    use_smote=True,
)
xgb_pipeline.fit(split.X_train, split.y_train)
xgb_metrics = evaluate_classifier(xgb_pipeline, split.X_test, split.y_test)
xgb_metrics

## Confusion Matrix and ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_estimator(
    xgb_pipeline,
    split.X_test,
    split.y_test,
    display_labels=["No Churn", "Churn"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("XGBoost Confusion Matrix")
RocCurveDisplay.from_estimator(xgb_pipeline, split.X_test, split.y_test, ax=axes[1])
axes[1].set_title("XGBoost ROC Curve")
plt.tight_layout()
plt.show()

## SHAP Explanations

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = save_shap_summary(xgb_pipeline, split.X_test, OUTPUT_DIR / "shap_summary.png", max_rows=250)
waterfall_path = save_shap_waterfall(xgb_pipeline, split.X_test, OUTPUT_DIR / "shap_waterfall.png")
summary_path, waterfall_path

In [ ]:
sample_customer = split.X_test.iloc[[0]]
matrix, feature_names = transformed_features(xgb_pipeline, sample_customer)
explainer = shap.Explainer(xgb_pipeline.named_steps["classifier"], matrix, feature_names=feature_names)
shap.plots.waterfall(explainer(matrix)[0], max_display=15)

## Save Final Models

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
baseline_path = save_pipeline(baseline_pipeline, MODEL_DIR / "logistic_regression_churn_model.pkl")
xgb_path = save_pipeline(xgb_pipeline, MODEL_DIR / "xgboost_churn_model.pkl")
baseline_path, xgb_path